In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("/sujin/PycharmProjects/rllm")

from pathlib import Path
from rllm.data import DatasetRegistry, Dataset
from rllm.eval.agent_loader import load_agent, register_agent, list_agents
from rllm.runner import Runner
from rllm.types import AgentConfig, Task
from rllm.eval.evaluator_loader import load_evaluator
from tqdm import tqdm
from utils.mpr import MultipleProcessRunnerSimplifier

In [81]:
dataset = DatasetRegistry.load_dataset("aime2024", "test")

item = dataset[0]
print(item.keys())   # → dict_keys(['question', 'answer', ...)
print(item["question"])

dict_keys(['id', 'problem', 'solution', 'answer', 'url', 'year', 'question', 'ground_truth', 'data_source'])
Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, the walk takes her 4 hours, including $t$ minutes spent in the coffee shop. When she walks $s+2$ kilometers per hour, the walk takes her 2 hours and 24 minutes, including $t$ minutes spent in the coffee shop. Suppose Aya walks at $s+\frac{1}{2}$ kilometers per hour. Find the number of minutes the walk takes her, including the $t$ minutes spent in the coffee shop.


In [82]:
agent = load_agent("react")
print(agent)

task = Task(
    id="gsm8k_1",
    instruction=item["question"],
    metadata=item,
    dataset_dir=Path("/root/.rllm/datasets/aime2024"),  # 相对路径
)

config = AgentConfig(
    base_url="https://api.minimaxi.com/v1",  # LiteLLM proxy 地址
    model="MiniMax-M2.7",
    session_uid="notebook-0",
)

setattr(config, "api_key", "sk-cp-6GAU5uz5mgdHd2eFAVj6GZYim6MdZQnROkO--M1h2a8uWAeL5vqxbLSoXYakkJzIsN0iuPy-iKpxd7JEFazYa2V8K0iez3B-ubZtgkf-C6nwGrdokCealrU")

runner = Runner(
    agent_flow=agent,
    evaluator_override=None,  # 或者传入 load_evaluator("math")
)

episode = agent.run(task, config)

In [83]:
print(episode.trajectories[0].output)

<think>
We need to parse the problem. Aya goes for a 9 km walk, then stops at a coffee shop (presumably after the walk). So total time for each scenario includes walking time plus coffee shop time (t minutes). We have two scenarios:

1. Walking at constant speed s km/h, the walk takes her 4 hours total (including t minutes coffee). So total time = 4 hours = 240 minutes = walking time + t. Walking time = total - coffee = 240 - t minutes.

Distance = 9 km. So walking time (in hours) = distance / speed = 9 / s hours. Convert to minutes: (9 / s) * 60 minutes = (540 / s) minutes.

Thus equation: (540 / s) + t = 240.

2. Walking at speed s + 2 km/h, total time = 2 hours 24 minutes = 2 + 24/60 = 2.4 hours = 144 minutes. So total minutes = 144 = walking time + t.

Walking time = 9 / (s + 2) hours = (9 * 60) / (s + 2) = 540 / (s + 2) minutes. So equation: (540 / (s + 2)) + t = 144.

We need to solve for s and t. Then find total time when speed = s + 0.5 km/h (i.e., s + 1/2). Let speed = s + 0.5

In [84]:
evaluator = load_evaluator("math_reward_fn")  # 或其他 registered evaluator
print(evaluator)
result = evaluator.evaluate(task, episode)

print(f"Reward: {result.reward}")
print(f"Is correct: {result.is_correct}")
print(f"Signals: {result.signals}")

Reward: 1.0
Is correct: True
Signals: [Signal(name='accuracy', value=1.0, metadata={})]


# Evaluate Qwen3-4B on GSM8K

In [14]:
register_agent("math", "cookbooks.math.sync_math_flow:math_flow")
list_agents()

[{'name': 'math',
  'source': 'registered',
  'description': '',
  'module': 'cookbooks.math.sync_math_flow:math_flow'},
 {'name': 'bash',
  'source': 'built-in',
  'description': 'Multi-turn ReAct bash loop inside a sandbox.',
  'module': 'rllm.harnesses.bash.BashHarness'},
 {'name': 'claude-code',
  'source': 'built-in',
  'description': 'Run the Claude Code CLI inside the sandbox.',
  'module': 'rllm.harnesses.claude_code.ClaudeCodeHarness'},
 {'name': 'react',
  'source': 'built-in',
  'description': 'One-shot LLM call. Default for data tasks (math, MCQ, QA).',
  'module': 'rllm.harnesses.react.ReActHarness'}]

In [30]:
dataset_id = "gsm8k"
dataset = DatasetRegistry.load_dataset(dataset_id, "test")
agent = load_agent("math")
evaluator = load_evaluator("math_reward_fn")  # 或其他 registered evaluator

def do(process_id, idx, item, writer):
    task = Task(
        id=dataset_id,
        instruction=item["question"],
        metadata=item,
        dataset_dir=Path(f"/root/.rllm/datasets/{dataset_id}"),  # 相对路径
    )

    config = AgentConfig(
        base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
        model="/sujin/Models/Qwen/Qwen3-4B",
        session_uid="notebook-0",
    )

    episode = agent.run(task, config)
    result = evaluator.evaluate(task, episode)

    content = episode.trajectories[0].steps[0].action
    print(len(content))

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("/sujin/Models/Qwen/Qwen3-4B")
    tokens = tokenizer.encode(content)
    print(len(tokens))

    writer.write(f"{int(result.is_correct)}\n")


items = [item for item in dataset][:1]
mprs = MultipleProcessRunnerSimplifier(items, do, n_process=1, return_results=True)
outputs = mprs.run()

[Errno 25] Inappropriate ioctl for device
Can't get terminal size, set terminal_y = None
812


/root/PycharmProjects/rllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


200
7Total: [##################################################] 100% 1/1 [00:00:05 < 00:00:00, 5.14s/it]8

Aggregating results...: 1it [00:00, 876.74it/s]


In [ ]:
results = [int(o) for o in outputs]
acc = sum(results) / len(results)
print(acc)

In [85]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")
# client = OpenAI(base_url="http://172.98.0.90:37863/v1", api_key="EMPTY")

messages: list[dict] = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "你好"},
]

resp = client.chat.completions.create(
    model="/sujin/Models/Qwen/Qwen3-4B",
    messages=messages,
    temperature=0.6,
    max_tokens=1000,
    max_completion_tokens=1000,
    timeout=100,
)
content = resp.choices[0].message.content or ""
print(content)

<think>
嗯，用户发来“你好”，这是一个很常见的问候语。首先，我需要回应这个问候，保持友好和自然。用户可能是在测试我的反应，或者只是想开始对话。我应该用简洁的中文回应，比如“你好！有什么我可以帮你的吗？”，这样既礼貌又直接。

接下来，我需要考虑用户可能的意图。他们可能想询问某个问题，或者只是需要一个聊天伙伴。我需要准备好提供帮助，但也要保持


In [53]:
dataset = DatasetRegistry.load_dataset("aime_2025", "train")
agent = load_agent("react")
evaluator = load_evaluator("math_reward_fn")  # 或其他 registered evaluator
item = dataset[0]

task = Task(
    id="aime2024",
    instruction=item["question"],
    metadata=item,
    dataset_dir=Path("/root/.rllm/datasets/aime_2025"),  # 相对路径
)

config = AgentConfig(
    base_url="http://localhost:30000/v1",  # LiteLLM proxy 地址
    model="/sujin/Models/Qwen/Qwen3-0.6B",
    session_uid="notebook-0",
)

episode = agent.run(task, config)
result = evaluator.evaluate(task, episode)


In [55]:
print(episode.metadata)
print(result)

{}
EvalOutput(reward=1.0, is_correct=True, signals=[Signal(name='accuracy', value=1.0, metadata={})], metadata={})
